In [ ]:
def process_image(path):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224, 224))
    return img



def extract_features(image):
    # Color
    mean = image.mean(axis=(0,1))
    std = image.std(axis=(0,1))

    # LBP
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    lbp = local_binary_pattern(gray, 8, 1, method="uniform")

    hist, _ = np.histogram(lbp.ravel(), bins=10, range=(0,10))
    hist = hist.astype("float")
    hist /= (hist.sum() + 1e-6)

    return np.concatenate([mean, std, hist])



import joblib
import cv2
from skimage.feature import local_binary_pattern
import streamlit as st
from PIL import Image
import numpy as np

model = joblib.load("model.pkl")
label_map = joblib.load("label_map.pkl")

reverse_map = {v:k for k,v in label_map.items()}
def extract_features(image):
    mean = image.mean(axis=(0,1))
    std = image.std(axis=(0,1))

    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    lbp = local_binary_pattern(gray, 8, 1, method="uniform")

    hist, _ = np.histogram(lbp.ravel(), bins=10, range=(0,10))
    hist = hist.astype("float")
    hist /= (hist.sum() + 1e-6)

    return np.concatenate([mean, std, hist])
treatments = {
    "healthy": "Plant is healthy, no treatment needed",
    "Leaf Spot": "Use resistant plants",
    "Gray Mold": "Improve ventilation",
    "Blossom Blight": "Apply fungicide",
    "Anthracnose Fruit Rot": "Remove infected fruits",
    "Angular Leafspot": "Use copper fungicide",
    "Powdery Mildew Fruit": "Apply sulfur spray",
    "Powdery Mildew Leaf": "Avoid humidity"
}


st.title("Strawberry Disease Detector 🍓")

file = st.file_uploader("Upload Image", type=["jpg", "jpeg", "png"])

if file is not None:
    try:
        image = Image.open(file).convert("RGB")
        st.image(image, caption="Uploaded Image")

        img = image.resize((224, 224))
        img = np.array(img)

        features = extract_features(img)

        probs = model.predict_proba([features])[0]
        pred = np.argmax(probs)

        confidence = probs[pred]
        disease = reverse_map[pred]

        st.success(f"🦠 Disease: {disease}")
        st.progress(float(confidence))
        st.write(f"Confidence: {confidence*100:.2f}%")

        st.info(f"💊 Treatment: {treatments.get(disease, 'No treatment found')}")

        if confidence < 0.6:
            st.warning("⚠️ Model not very confident, try another image")

    except Exception as e:
        st.error(f"Error: {e}")







                # Confidence display
        if confidence > 0.8:
            st.success(f"High confidence: {confidence*100:.2f}%")
        elif confidence > 0.6:
            st.warning(f"Medium confidence: {confidence*100:.2f}%")
        else:
            st.info(f"Low confidence: {confidence*100:.2f}%")

        st.progress(float(confidence))